# Customer Lifecycle Metrics ETL

## Purpose
Provide real-time overall customer lifecycle KPIs (total customers, average orders per customer, average lifetime value) for executive dashboards.

## Input
* **Source:** `big_data.silver.orders`
* **Source:** `big_data.silver.order_products`
* **Source:** `big_data.silver.products_enriched`

## Output
* **Target:** `big_data.gold.vw_customer_lifecycle_metrics`
* **Refresh:** Real-time (always reflects current Silver data)

## SQL Logic
1. CTE: JOIN orders, order_products, products_enriched to calculate per-customer metrics
2. GROUP BY user_id to get total_orders and lifetime_value_usd per customer
3. Aggregate across all customers: COUNT customers, AVG orders, AVG lifetime value
4. Single row result with 3 KPIs

In [0]:
%sql
-- Customer Lifecycle Metrics View
-- Purpose: Real-time customer KPIs for executive dashboards

CREATE OR REPLACE VIEW big_data.gold.vw_customer_lifecycle_metrics AS
WITH customer_metrics AS (
  SELECT 
    o.user_id,
    COUNT(DISTINCT o.order_id) AS total_orders,
    ROUND(SUM(p.price_usd), 2) AS lifetime_value_usd
  FROM big_data.silver.orders o
  JOIN big_data.silver.order_products op ON o.order_id = op.order_id
  JOIN big_data.silver.products_enriched p ON op.product_id = p.product_id
  GROUP BY o.user_id
)
SELECT 
  COUNT(DISTINCT user_id) AS total_customers,
  ROUND(AVG(total_orders), 2) AS avg_orders_per_customer,
  ROUND(AVG(lifetime_value_usd), 2) AS avg_customer_lifetime_value_usd
FROM customer_metrics;

In [0]:
%sql
-- Verify view exists and preview KPIs
-- Returns single row with 3 executive metrics

SELECT * FROM big_data.gold.vw_customer_lifecycle_metrics;